## 📊 Notebook-Übung: Tabellen verknüpfen mit SQL JOINs

:::{.callout-note}
## Wie man dieses Notebook lokal ausführt

1. Erstelle einen Ordner
2. Erstelle eine virtuelle Python-Umgebung  
   `python -m venv .venv`
3. Aktiviere die virtuelle Umgebung  
   `.venv\Scripts\activate`
4. Installiere Jupyter und pandas  
   `pip install jupyter pandas`
5. Lade diese Datei in deinen Ordner herunter
6. Starte den lokalen Jupyter-Server  
   `jupyter-lab`
:::

### Kontext

In der realen Welt sind Daten fast nie in einer einzigen Tabelle gespeichert.
Kundendaten liegen getrennt von Bestelldaten, Klassenlisten getrennt von Noten.
Um sinnvolle Analysen zu machen, müssen wir Tabellen **verknüpfen** — das
nennt man einen **JOIN**.

### Das Wichtigste auf einen Blick

| Was ich will | SQL |
|---|---|
| Nur übereinstimmende Zeilen | `SELECT ... FROM A INNER JOIN B ON A.key = B.key` |
| Alle Zeilen der linken Tabelle | `SELECT ... FROM A LEFT JOIN B ON A.key = B.key` |
| Fehlende Einträge finden | `... LEFT JOIN ... WHERE B.spalte IS NULL` |

**Dauer:** ca. 40–50 Minuten

**Voraussetzungen:** SQL-Grundlagen (`SELECT`, `FROM`, `WHERE`, Aggregatfunktionen), grundlegende Python-Kenntnisse

---

### Setup

Führe die folgende Zelle aus, um die Datenbank vorzubereiten. **Nicht verändern.**

In [ ]:
# --- Setup (nicht verändern) ---
import sqlite3
import pandas as pd

# Hilfsfunktion: Führt eine SQL-Abfrage aus und gibt ein DataFrame zurück.
def sql(query):
    cursor.execute(query)
    cols = [d[0] for d in cursor.description]
    return pd.DataFrame(cursor.fetchall(), columns=cols)

# Erstelle eine SQLite-Datenbank im Arbeitsspeicher (RAM)
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

cursor.executescript("""
-- Tabellen für Aufgaben 1-2: Die Prüfungsliste
CREATE TABLE klassenliste (
    schueler_id INTEGER PRIMARY KEY,
    name        TEXT,
    klasse      TEXT
);
CREATE TABLE noten (
    schueler_id INTEGER,
    note        REAL,
    FOREIGN KEY (schueler_id) REFERENCES klassenliste(schueler_id)
);
INSERT INTO klassenliste VALUES
    (1, 'Lea',   '2fP'),
    (2, 'Jan',   '2fP'),
    (3, 'Mira',  '2fP'),
    (4, 'Noah',  '2fP'),
    (5, 'Elena', '2fP'),
    (6, 'Luca',  '2fP');
INSERT INTO noten VALUES
    (1, 5.5),
    (2, 4.0),
    (5, 5.0);

-- Tabellen für Aufgabe 3: Der Online-Shop (gleiche Daten wie im SQL-Aggregatfunktionen-Notebook!)
CREATE TABLE products (
    product_id   INTEGER PRIMARY KEY,
    product_name TEXT,
    category     TEXT,
    price        REAL
);
CREATE TABLE orders (
    order_id    INTEGER PRIMARY KEY,
    product_id  INTEGER,
    quantity    INTEGER,
    customer    TEXT,
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);
INSERT INTO products VALUES
    (1, 'Laptop',  'Electronics', 1200.00),
    (2, 'Mouse',   'Electronics',   25.00),
    (3, 'Desk',    'Furniture',    450.00),
    (4, 'Chair',   'Furniture',    320.00),
    (5, 'Headset', 'Electronics',   80.00);
INSERT INTO orders VALUES
    (1, 1, 2, 'Anna'),
    (2, 2, 5, 'Ben'),
    (3, 3, 1, 'Clara'),
    (4, 1, 1, 'Daniel'),
    (5, 4, 3, 'Anna'),
    (6, 5, 2, 'Ben'),
    (7, 2, 8, 'Clara'),
    (8, 3, 2, 'Evi'),
    (9, 5, 1, 'Anna');

-- Tabellen für Aufgabe 4: Sportverein
CREATE TABLE mitglieder (
    mitglied_id INTEGER PRIMARY KEY,
    name        TEXT,
    sportart    TEXT
);
CREATE TABLE turnier_ergebnisse (
    mitglied_id INTEGER,
    turnier     TEXT,
    rang        INTEGER,
    FOREIGN KEY (mitglied_id) REFERENCES mitglieder(mitglied_id)
);
INSERT INTO mitglieder VALUES
    (1, 'Anna',   'Fussball'),
    (2, 'Ben',    'Tennis'),
    (3, 'Chiara', 'Fussball'),
    (4, 'David',  'Schwimmen'),
    (5, 'Eva',    'Tennis'),
    (6, 'Finn',   'Fussball');
INSERT INTO turnier_ergebnisse VALUES
    (1, 'Kantonalmeisterschaft', 2),
    (2, 'Vereinsturnier', 5),
    (3, 'Kantonalmeisterschaft', 8),
    (5, 'Vereinsturnier', 3);

-- Tabellen für Aufgabe 5: Bibliothek
CREATE TABLE buecher (
    buch_id INTEGER PRIMARY KEY,
    titel   TEXT,
    autor   TEXT,
    genre   TEXT
);
CREATE TABLE ausleihen (
    ausleihe_id INTEGER PRIMARY KEY,
    buch_id     INTEGER,
    schueler    TEXT,
    datum       TEXT,
    FOREIGN KEY (buch_id) REFERENCES buecher(buch_id)
);
INSERT INTO buecher VALUES
    (1, 'Harry Potter', 'J.K. Rowling', 'Fantasy'),
    (2, 'Gregs Tagebuch', 'Jeff Kinney', 'Humor'),
    (3, 'Die drei ???', 'Verschiedene', 'Krimi'),
    (4, 'Eragon', 'C. Paolini', 'Fantasy'),
    (5, 'Tintenherz', 'C. Funke', 'Fantasy'),
    (6, 'Rico, Oskar und die Tieferschatten', 'A. Steinhöfel', 'Krimi');
INSERT INTO ausleihen VALUES
    (1, 1, 'Lea',   '2026-03-10'),
    (2, 2, 'Jan',   '2026-03-12'),
    (3, 1, 'Mira',  '2026-03-15'),
    (4, 3, 'Noah',  '2026-03-18'),
    (5, 2, 'Elena', '2026-03-20');
""")
conn.commit()
print('Datenbank bereit. Viel Erfolg!')

---

### Aufgabe 1: INNER JOIN — Wer hat die Prüfung geschrieben?

**Tabellen:**

| Tabelle | Spalten |
|---|---|
| `klassenliste` | `schueler_id`, `name`, `klasse` |
| `noten` | `schueler_id`, `note` |

Frau Müller will nur die Schüler sehen, die die Prüfung **tatsächlich
geschrieben haben**, zusammen mit ihren Namen und Noten.

Schreibe ein `SELECT` mit `INNER JOIN`, um `klassenliste` und `noten`
über die Spalte `schueler_id` zu verknüpfen. Zeige `name` und `note`.

**Erwartetes Ergebnis:** 3 Zeilen (Lea, Jan, Elena).

In [ ]:
# AUFGABE 1: INNER JOIN von klassenliste und noten.
# Nutze sql(""" ... """) um deine Abfrage auszuführen.

result1 = sql("""
-- Deine Abfrage hier
""")
print('Aufgabe 1:\n', result1)

<details>
<summary>Lösung anzeigen</summary>

```sql
SELECT klassenliste.name, noten.note
FROM klassenliste
INNER JOIN noten
  ON klassenliste.schueler_id = noten.schueler_id;
```

Ergebnis: 3 Zeilen — nur Lea (5.5), Jan (4.0) und Elena (5.0), weil nur diese in **beiden** Tabellen vorkommen.

</details>

### Aufgabe 2: LEFT JOIN — Die vollständige Übersicht

Frau Müller braucht **alle 6 Schüler** in ihrer Übersicht — auch die,
die nicht an der Prüfung teilgenommen haben. Bei fehlenden Noten
soll `NULL` stehen.

**2a)** Schreibe ein `SELECT` mit `LEFT JOIN`, um alle Schüler mit ihren
Noten zu sehen.

**2b)** Erweitere die Abfrage mit einer `WHERE`-Klausel, um nur die
Schüler zu finden, die **keine Note** haben (`IS NULL`).

**2c)** Beantworte: Warum stehen dort `NULL`-Werte?

In [ ]:
# AUFGABE 2a: LEFT JOIN von klassenliste und noten.

result2a = sql("""
-- Deine Abfrage hier
""")
print('Aufgabe 2a:\n', result2a)

<details>
<summary>Lösung anzeigen (2a)</summary>

```sql
SELECT klassenliste.name, noten.note
FROM klassenliste
LEFT JOIN noten
  ON klassenliste.schueler_id = noten.schueler_id;
```

Ergebnis: 6 Zeilen. Mira, Noah und Luca haben `None` (= NULL) in der Notenspalte, weil sie nicht in der `noten`-Tabelle vorkommen.

</details>

In [ ]:
# AUFGABE 2b: Nur Schüler OHNE Note finden (LEFT JOIN + WHERE IS NULL).

result2b = sql("""
-- Deine Abfrage hier
""")
print('Aufgabe 2b:\n', result2b)

<details>
<summary>Lösung anzeigen (2b)</summary>

```sql
SELECT klassenliste.name
FROM klassenliste
LEFT JOIN noten
  ON klassenliste.schueler_id = noten.schueler_id
WHERE noten.note IS NULL;
```

Ausgabe: Mira, Noah, Luca — die drei Schüler, die krank waren und die Prüfung nicht geschrieben haben.

</details>

**Aufgabe 2c:** Warum stehen bei diesen Schülern `NULL`-Werte?

*Deine Antwort:*

---

### Aufgabe 3: Online-Shop — JOIN + GROUP BY

**Tabellen:**

| Tabelle | Spalten |
|---|---|
| `products` | `product_id`, `product_name`, `category`, `price` |
| `orders` | `order_id`, `product_id`, `quantity`, `customer` |

Du kennst diese Tabellen bereits vom SQL-Aggregatfunktionen-Notebook.
Jetzt verknüpfen wir sie mit einem JOIN!

**3a)** Schreibe einen `INNER JOIN`, um jede Bestellung mit dem
zugehörigen Produktnamen und Preis anzuzeigen.
Zeige: `product_name`, `quantity`, `price`, `customer`.

**3b)** Berechne den **Gesamtumsatz pro Kategorie**.
Tipp: Umsatz = `price * quantity`. Nutze `SUM()` und `GROUP BY`.

In [ ]:
# AUFGABE 3a: INNER JOIN von orders und products.

result3a = sql("""
-- Deine Abfrage hier
""")
print('Aufgabe 3a:\n', result3a)

<details>
<summary>Lösung anzeigen (3a)</summary>

```sql
SELECT products.product_name, orders.quantity, products.price, orders.customer
FROM orders
INNER JOIN products
  ON orders.product_id = products.product_id;
```

Ergebnis: 9 Zeilen — jede Bestellung jetzt mit Produktname und Preis.

</details>

In [ ]:
# AUFGABE 3b: Gesamtumsatz pro Kategorie (JOIN + GROUP BY).

result3b = sql("""
-- Deine Abfrage hier
""")
print('Aufgabe 3b:\n', result3b)

<details>
<summary>Lösung anzeigen (3b)</summary>

```sql
SELECT products.category,
       SUM(orders.quantity * products.price) AS umsatz
FROM orders
INNER JOIN products
  ON orders.product_id = products.product_id
GROUP BY products.category;
```

Ergebnis: Electronics = 4165.00, Furniture = 2310.00.

</details>

---

### Aufgabe 4: Sportverein — Wer hat (nicht) am Turnier teilgenommen?

**Tabellen:**

| Tabelle | Spalten |
|---|---|
| `mitglieder` | `mitglied_id`, `name`, `sportart` |
| `turnier_ergebnisse` | `mitglied_id`, `turnier`, `rang` |

Der Sportverein will eine Übersicht über alle Mitglieder und ihre
Turnierergebnisse. Nicht alle Mitglieder haben an einem Turnier
teilgenommen!

**4a)** Zeige alle Mitglieder mit ihren Turnierergebnissen.
Auch Mitglieder ohne Turnierergebnis sollen erscheinen.
Welchen JOIN-Typ brauchst du?

**4b)** Finde heraus: Welche Mitglieder haben noch **nie** an einem
Turnier teilgenommen?

In [ ]:
# AUFGABE 4a: Alle Mitglieder mit Turnierergebnissen (auch ohne!).

result4a = sql("""
-- Deine Abfrage hier
""")
print('Aufgabe 4a:\n', result4a)

<details>
<summary>Lösung anzeigen (4a)</summary>

```sql
SELECT mitglieder.name, mitglieder.sportart,
       turnier_ergebnisse.turnier, turnier_ergebnisse.rang
FROM mitglieder
LEFT JOIN turnier_ergebnisse
  ON mitglieder.mitglied_id = turnier_ergebnisse.mitglied_id;
```

Ergebnis: 6 Zeilen. David und Finn haben `NULL` bei Turnier und Rang.

</details>

In [ ]:
# AUFGABE 4b: Mitglieder, die NIE an einem Turnier teilgenommen haben.

result4b = sql("""
-- Deine Abfrage hier
""")
print('Aufgabe 4b:\n', result4b)

<details>
<summary>Lösung anzeigen (4b)</summary>

```sql
SELECT mitglieder.name, mitglieder.sportart
FROM mitglieder
LEFT JOIN turnier_ergebnisse
  ON mitglieder.mitglied_id = turnier_ergebnisse.mitglied_id
WHERE turnier_ergebnisse.turnier IS NULL;
```

Ergebnis: David (Schwimmen) und Finn (Fussball) — sie haben kein Turnierergebnis.

</details>

---

### Aufgabe 5: Bibliothek — Welche Bücher wurden nie ausgeliehen?

**Tabellen:**

| Tabelle | Spalten |
|---|---|
| `buecher` | `buch_id`, `titel`, `autor`, `genre` |
| `ausleihen` | `ausleihe_id`, `buch_id`, `schueler`, `datum` |

Die Schulbibliothek will herausfinden, welche Bücher noch nie
ausgeliehen wurden, um sie besser zu bewerben.

**5a)** Zeige alle Bücher mit ihren Ausleihen an (auch Bücher
ohne Ausleihe).

**5b)** Finde die Bücher, die **noch nie** ausgeliehen wurden.

**5c)** Zähle, wie oft jedes Buch ausgeliehen wurde.
Sortiere absteigend nach der Ausleihzahl.
Tipp: Nutze `COUNT()`, `GROUP BY` und `ORDER BY`.

In [ ]:
# AUFGABE 5a: Alle Bücher mit ihren Ausleihen (auch ohne!).

result5a = sql("""
-- Deine Abfrage hier
""")
print('Aufgabe 5a:\n', result5a)

<details>
<summary>Lösung anzeigen (5a)</summary>

```sql
SELECT buecher.titel, buecher.autor,
       ausleihen.schueler, ausleihen.datum
FROM buecher
LEFT JOIN ausleihen
  ON buecher.buch_id = ausleihen.buch_id;
```

Ergebnis: 8 Zeilen. Eragon, Tintenherz und Rico haben `NULL` bei Schüler und Datum.

</details>

In [ ]:
# AUFGABE 5b: Bücher, die noch NIE ausgeliehen wurden.

result5b = sql("""
-- Deine Abfrage hier
""")
print('Aufgabe 5b:\n', result5b)

<details>
<summary>Lösung anzeigen (5b)</summary>

```sql
SELECT buecher.titel, buecher.autor
FROM buecher
LEFT JOIN ausleihen
  ON buecher.buch_id = ausleihen.buch_id
WHERE ausleihen.ausleihe_id IS NULL;
```

Ergebnis: Eragon, Tintenherz und Rico, Oskar und die Tieferschatten — diese drei wurden noch nie ausgeliehen.

</details>

In [ ]:
# AUFGABE 5c: Wie oft wurde jedes Buch ausgeliehen? (JOIN + GROUP BY + ORDER BY)

result5c = sql("""
-- Deine Abfrage hier
""")
print('Aufgabe 5c:\n', result5c)

<details>
<summary>Lösung anzeigen (5c)</summary>

```sql
SELECT buecher.titel,
       COUNT(ausleihen.ausleihe_id) AS anzahl_ausleihen
FROM buecher
LEFT JOIN ausleihen
  ON buecher.buch_id = ausleihen.buch_id
GROUP BY buecher.titel
ORDER BY anzahl_ausleihen DESC;
```

Ergebnis: Harry Potter und Gregs Tagebuch je 2×, Die drei ??? 1×, die restlichen drei 0×.

</details>

---

### Bonus: JOIN über drei Tabellen

Für diese Aufgabe erstellen wir eine zusätzliche Tabelle.
Führe zuerst die nächste Zelle aus.

In [ ]:
# Bonus-Setup (nicht verändern)
cursor.executescript("""
CREATE TABLE hausaufgaben (
    schueler_id INTEGER,
    punkte      INTEGER,
    FOREIGN KEY (schueler_id) REFERENCES klassenliste(schueler_id)
);
INSERT INTO hausaufgaben VALUES
    (1, 18),
    (2, 12),
    (3, 20),
    (4, 15),
    (5, 17),
    (6, 10);
""")
conn.commit()
print('Bonus-Tabelle erstellt!')

**Tabellen:**

| Tabelle | Spalten |
|---|---|
| `klassenliste` | `schueler_id`, `name`, `klasse` |
| `noten` | `schueler_id`, `note` |
| `hausaufgaben` | `schueler_id`, `punkte` |

Frau Müller will jetzt eine **Gesamtübersicht**: Name, Prüfungsnote
UND Hausaufgabenpunkte — für alle 6 Schüler.

Tipp: Du kannst zwei JOINs hintereinander schreiben:
```sql
SELECT ...
FROM tabelle_a
LEFT JOIN tabelle_b ON ...
LEFT JOIN tabelle_c ON ...;
```

In [ ]:
# BONUS: JOIN über drei Tabellen.

bonus = sql("""
-- Deine Abfrage hier
""")
print('Bonus:\n', bonus)

<details>
<summary>Lösung anzeigen</summary>

```sql
SELECT klassenliste.name, noten.note, hausaufgaben.punkte
FROM klassenliste
LEFT JOIN noten
  ON klassenliste.schueler_id = noten.schueler_id
LEFT JOIN hausaufgaben
  ON klassenliste.schueler_id = hausaufgaben.schueler_id;
```

Ergebnis: 6 Zeilen. Alle Schüler haben Hausaufgabenpunkte, aber nur 3 haben Noten. Bei den Kranken steht `NULL` in der Notenspalte.

</details>

---

## Reflexion

Beantworte die folgenden Fragen in dieser Zelle:

1. Erkläre in eigenen Worten: Was ist der Unterschied zwischen einem
   `INNER JOIN` und einem `LEFT JOIN`?

   *Deine Antwort:*

2. Nenne eine Situation aus deinem Alltag, in der du zwei Listen
   zusammenführen müsstest. Welchen JOIN-Typ würdest du verwenden?

   *Deine Antwort:*

3. In Aufgabe 3 hast du einen JOIN mit GROUP BY kombiniert.
   Warum brauchst du zuerst den JOIN, bevor du gruppieren kannst?

   *Deine Antwort:*